In [2]:
# Importaciones y configuración global
import os
import warnings

import numpy as np
import pandas as pd
import torch
import sentencepiece
import google.protobuf
import sys
import transformers

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score)
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    EarlyStoppingCallback,
    GPT2ForSequenceClassification,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline as hf_pipeline,
    XLMRobertaTokenizer
)

from transformers.utils import is_protobuf_available

In [3]:
warnings.filterwarnings("ignore")
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

Al hacer esto, estoy forzando que use mi CPU y no mi GPU, 
Ya que, en este caso mi CPU es más potente que mi GPU

In [4]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(f"Dispositivo: {device}")
#if device.type == "cuda":
#    print(f"GPU: {torch.cuda.get_device_name(0)}")


device = torch.device("cpu")
print(f"Dispositivo: {device}")
torch.set_num_threads(2)
torch.set_num_interop_threads(2)

Dispositivo: cpu


##### Ojo
Esto anterior es muy importante, como sabemos lo más ideal, como ya sabemos la GPU es lo más ideal para esta clase de labores, pero tambien se puede hacer con CPU, el problema es que si se hace con CPU puede tarar de minimo media a incluso una hora pudiendose alargar a mucho más (incluso hasta días enteros) esto dependiendo de la candidad de RAM que se tiene.

Esta pensado para que si esto llega a una persona que desconosca su equipo, pueda identificar en que situación se encuentra.

## Carga del dataset

In [5]:
directorio_base = os.getcwd()
sub_carpeta = 'datasets'
nombre_archivo = 'sentiment_analysis_dataset_simulacion.csv'
ruta_entrada = os.path.join(directorio_base, sub_carpeta, nombre_archivo)

if not os.path.exists(ruta_entrada):
    print(f"No se encontró el archivo en la ruta: {ruta_entrada}")
else: 
    print("Archivo encontrado con éxito.")

ruta_salida = f'resultado_predicciones_{nombre_archivo}.csv'
columna_de_texto = 'text'
df = pd.read_csv(f'{ruta_entrada}')


print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(df.head(3))

Archivo encontrado con éxito.
Dataset cargado: 2,584 filas × 7 columnas
           user                                               text  \
0    @erreborda               termine bien abrumado después de hoy   
1  @shpiderduck                                 me siento abrumado   
2   @Alex_R_art  Me siento un poco abrumado por la cantidad de ...   

                         date      emotion sentiment  Negative Categoria  
0   Jan 6, 2024 · 2:53 AM UTC  overwhelmed    scared         1  Deportes  
1   Jan 6, 2024 · 2:35 AM UTC  overwhelmed    scared         1  Deportes  
2  Jan 6, 2024 · 12:20 AM UTC  overwhelmed    scared         1  Deportes  


# [! NOTA]
Estoy asuminedo que las columnas del dataset vienen explicitamente con las columnas:
- text
- Negative (Esto debido a mi enfoque que le voya a dar, que prefiero enfocarme en predecir los de carácter negativo, que los psoitivos)
- Categoria (la categoría periodística, aunque si hay un nombre técnico para esto sería mejor)
- PAra mejorar o adaptar esto, deberia de hablar con el cliente o mi objetivo para mejorar estos aspectos

In [6]:

# Nombres de columnas esperadas del dataset de entrenamiento
comment_col = 'text'       # columna con el texto del comentario
label_col = 'Negative'     # 0 = positivo, 1 = negativo
category_col = 'Categoria' # categoría periodística

# Verificar existencia
for col in [comment_col, label_col, category_col]:
    assert col in df.columns, (
        f"Error: la columna '{col}' no fue encontrada. "
        f"Columnas disponibles: {list(df.columns)}"
    )

print("Distribución de clases")
conteo = df[label_col].value_counts()
total = len(df)
print(f"Positivos (0): {conteo.get(0, 0):,}  ({conteo.get(0, 0) / total * 100:.1f}%)")
print(f"Negativos (1): {conteo.get(1, 0):,}  ({conteo.get(1, 0) / total * 100:.1f}%)")

print("\nCategorías periodísticas")
print(df[category_col].value_counts().to_string())

print("\nLongitud de comentarios (palabras)")
df['_len'] = df[comment_col].astype(str).str.split().str.len()
print(df['_len'].describe().round(1))

Distribución de clases
Positivos (0): 1,425  (55.1%)
Negativos (1): 1,159  (44.9%)

Categorías periodísticas
Categoria
Política     1440
Deportes      940
Farándula     204

Longitud de comentarios (palabras)
count    2584.0
mean       23.5
std        18.7
min         1.0
25%        11.0
50%        20.0
75%        35.0
max       585.0
Name: _len, dtype: float64


La parte de limpieza se va a quitar, debido a que ya hay una API que limpia los datos. E incluso podemos aspirar a realizar microservicios.

In [10]:
# Limpieza del dataset
df_clean = df[[comment_col, label_col, category_col]].copy()
antes = len(df_clean)

# Eliminar nulos y comentarios vacíos
df_clean = df_clean.dropna()
df_clean[comment_col] = df_clean[comment_col].astype(str).str.strip()
df_clean = df_clean[df_clean[comment_col].str.len() > 2]

# Normalizar etiqueta
df_clean[label_col] = df_clean[label_col].astype(int)
df_clean = df_clean[df_clean[label_col].isin([0, 1])]

# Normalizar categoría: mayúsculas y sin espacios extra
df_clean[category_col] = (
    df_clean[category_col]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
)

print(f"Registros antes: {antes:,}")
print(f"Registros después: {len(df_clean):,}  (eliminados: {antes - len(df_clean):,})")
print(f"Categorías normalizadas: {sorted(df_clean[category_col].unique())}")

Registros antes: 2,584
Registros después: 2,584  (eliminados: 0)
Categorías normalizadas: ['DEPORTES', 'FARÁNDULA', 'POLÍTICA']


In [11]:
def construir_texto(row):
    categoria = row[category_col]
    comentario = row[comment_col]
    return f"[{categoria}] {comentario}"

df_clean['texto_modelo'] = df_clean.apply(construir_texto, axis=1)

print("Ejemplos de texto con prefijo de categoría")
for _, row in df_clean.sample(3, random_state=seed).iterrows():
    etiqueta = "NEG" if row[label_col] == 1 else "POS"
    print(f"[{etiqueta}] {row['texto_modelo'][:90]}...")

Ejemplos de texto con prefijo de categoría
[POS] [POLÍTICA] Esas brillantes e ineditas ideas que tienen los funcionarios uribistas antioque...
[NEG] [DEPORTES] Mirada posó en Chris y, exasperado, le dió la linterna.  — Sostenga eso. — orde...
[POS] [POLÍTICA] Como amé esta semana sin preocupaciones, sin estrés, solo despertarme, desayuna...


##### Nota
Lo anterior dependera de la calidad del dataset

In [12]:
# División Train / Validación / Test (70/15/15) estratificada
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=seed,
    stratify=df_clean[label_col],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=seed,
    stratify=temp_df[label_col],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,} registros")
print(f"Validación: {len(val_df):,} registros")
print(f"Test: {len(test_df):,} registros")

Train: 1,808 registros
Validación: 388 registros
Test: 388 registros


In [13]:
for nombre, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    pos = (split[label_col] == 0).sum()
    neg = (split[label_col] == 1).sum()
    print(f"{nombre:6s} -> pos: {pos} | neg: {neg}")

Train  -> pos: 997 | neg: 811
Val    -> pos: 214 | neg: 174
Test   -> pos: 214 | neg: 174


In [14]:
# Tokenizador GPT-2 en español
model_name = "datificate/gpt2-small-spanish"
max_len = 128

print(f"Cargando tokenizador '{model_name}'...")
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizador listo - vocab size: {tokenizer.vocab_size:,}")

ejemplo = df_clean[category_col].unique()[:3]
for cat in ejemplo:
    tok = tokenizer.tokenize(f"[{cat}]")
    print(f"[{cat}] -> {tok}")

Cargando tokenizador 'datificate/gpt2-small-spanish'...
Tokenizador listo - vocab size: 50,257
[DEPORTES] -> ['[', 'DE', 'P', 'OR', 'T', 'ES', ']']
[POLÍTICA] -> ['[', 'P', 'OL', 'Ãį', 'TI', 'CA', ']']
[FARÁNDULA] -> ['[', 'FA', 'R', 'Ãģ', 'N', 'D', 'U', 'LA', ']']


In [15]:
# Dataset de PyTorch
class ComentariosDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.labels = torch.tensor(labels.values, dtype=torch.long)
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt',
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

print("Tokenizando datasets...")
train_dataset = ComentariosDataset(train_df['texto_modelo'], train_df[label_col], tokenizer, max_len)
val_dataset = ComentariosDataset(val_df['texto_modelo'], val_df[label_col], tokenizer, max_len)
test_dataset = ComentariosDataset(test_df['texto_modelo'], test_df[label_col], tokenizer, max_len)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Tokenizando datasets...
Train: 1808 | Val: 388 | Test: 388


In [16]:
# Modelo GPT-2 con cabeza de clasificación binaria
print(f"Cargando modelo '{model_name}'...")
model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
model.config.pad_token_id = tokenizer.eos_token_id
model.config.id2label = {0: "Positivo", 1: "Negativo"}
model.config.label2id = {"Positivo": 0, "Negativo": 1}
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modelo listo - parámetros entrenables: {trainable:,} (fine-tuning completo)")


Cargando modelo 'datificate/gpt2-small-spanish'...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at datificate/gpt2-small-spanish and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelo listo - parámetros entrenables: 124,441,344 (fine-tuning completo)


In [17]:
# Hiperparámetros y métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted'),
        "f1_pos": f1_score(labels, preds, pos_label=0, average='binary'),
        "f1_neg": f1_score(labels, preds, pos_label=1, average='binary'),
    }

epochs = 7
lr = 1e-5
batch = 16
eval_batch = 32

training_args = TrainingArguments(
    output_dir='./checkpoints_gpt2_es_v2',
    num_train_epochs=epochs,
    per_device_train_batch_size=batch,
    per_device_eval_batch_size=eval_batch,
    learning_rate=lr,
    weight_decay=0.01,
    warmup_ratio=0.15,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    seed=seed,
    report_to='none',
)

print("Hiperparámetros configurados")
print(f"epochs={epochs} | lr={lr} | batch={batch} | fp16={torch.cuda.is_available()}")

Hiperparámetros configurados
epochs=7 | lr=1e-05 | batch=16 | fp16=False


##### Hiperparámetros ajustados (planeado para dataset de 2,000 – 10,000 muestras)

Investigue y con datasets de este tamaño conviene:
- Más epochs (7) para que el modelo aprenda bien
- LR bajo (1e-5) para no sobreajustar el fine-tuning completo
- Warmup generoso (15%) para estabilizar el inicio
- Early stopping con paciencia=3 (3 epochs sin mejorar → detener)
- Weight decay moderado (0.01) como regularización

In [13]:
# Entrenamiento
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Iniciando entrenamiento...")
trainer.train()
print("Entrenamiento finalizado")

Iniciando entrenamiento...


Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Pos,F1 Neg
1,0.633407,0.293549,0.909794,0.908429,0.923747,0.889590
2,0.264172,0.240129,0.917526,0.916450,0.929825,0.900000
3,0.239128,0.289773,0.922680,0.921566,0.934498,0.905660
4,0.189453,0.271027,0.922680,0.921566,0.934498,0.905660
5,0.156319,0.255142,0.922680,0.921672,0.934211,0.906250
6,0.126162,0.243462,0.920103,0.919313,0.931264,0.904615
7,0.122990,0.250910,0.922680,0.921773,0.933921,0.906832


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]


Entrenamiento finalizado


In [14]:


# El modelo GPT-2 ya esta entrenado (en disco)
# No necesitas hacer nada, 'model' y 'tokenizer' ya están disponibles.

# Otra Opcion: cargarlo desde el checkpoint guardado (sesión nueva)

# from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
# import torch
#
# save_path = './modelo_sentimientos_gpt2_es_v2'   # ajusta si lo moviste
# device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# tokenizer = GPT2Tokenizer.from_pretrained(save_path)
# model     = GPT2ForSequenceClassification.from_pretrained(save_path)
# tokenizer.pad_token = tokenizer.eos_token
# model.config.pad_token_id = tokenizer.eos_token_id
# model.to(device)
# model.eval()
# max_len = 128
# print(" Modelo GPT-2 cargado desde disco")



In [26]:
trainer.save_model("./modelo_final")

os.listdir("./modelo_final")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


['model.safetensors', 'config.json', 'training_args.bin']

In [25]:
ls -lah {training_args.output_dir}

total 36K
drwxrwxr-x 9 papiit papiit 4.0K jun  4 19:56 ./
drwxrwxr-x 7 papiit papiit 4.0K jun  4 20:23 ../
drwxrwxr-x 2 papiit papiit 4.0K jun  4 17:14 checkpoint-113/
drwxrwxr-x 2 papiit papiit 4.0K jun  4 17:41 checkpoint-226/
drwxrwxr-x 2 papiit papiit 4.0K jun  4 18:08 checkpoint-339/
drwxrwxr-x 2 papiit papiit 4.0K jun  4 18:35 checkpoint-452/
drwxrwxr-x 2 papiit papiit 4.0K jun  4 19:02 checkpoint-565/
drwxrwxr-x 2 papiit papiit 4.0K jun  4 19:29 checkpoint-678/
drwxrwxr-x 2 papiit papiit 4.0K jun  4 19:56 checkpoint-791/


In [29]:
tokenizer.save_pretrained("./modelo_final")
os.listdir("./modelo_final")

['model.safetensors',
 'config.json',
 'tokenizer.json',
 'tokenizer_config.json',
 'training_args.bin']

In [18]:

MODELO_PATH = "./modelo_final"

tokenizer = AutoTokenizer.from_pretrained(MODELO_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODELO_PATH
)

Asegurarnos que esta presente para lo que viene a continuación

In [19]:
print(sentencepiece.__version__)
print(sys.version)
print(google.protobuf.__version__)
print(transformers.__version__)

0.2.1
protobuf OK
3.13.13 (main, Apr 20 2026, 15:26:12) [GCC 9.4.0]


In [20]:
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("Protobuf:", google.protobuf.__version__)

3.20.3
4.52.4
Python: 3.13.13 (main, Apr 20 2026, 15:26:12) [GCC 9.4.0]
Torch: 2.12.0+cu130
Transformers: 4.52.4
SentencePiece: 0.2.1
Protobuf: 3.20.3


In [21]:
tokenizer = AutoTokenizer.from_pretrained(
    "joeddav/xlm-roberta-large-xnli"
)

print("Tokenizer cargado")

Tokenizer cargado


In [22]:
print(model.__class__)

<class 'transformers.models.gpt2.modeling_gpt2.GPT2ForSequenceClassification'>


In [23]:
print("Modelo GPT-2 listo para inferencia")

print("Cargando tokenizador lento de XLM-RoBERTa...")
tokenizer_categoria = AutoTokenizer.from_pretrained(
    "joeddav/xlm-roberta-large-xnli",
    use_fast=False
)

print("Cargando clasificador zero-shot (XLM-RoBERTa)...")
clasificador_categoria = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    tokenizer=tokenizer_categoria,
    device=0 if torch.cuda.is_available() else -1,
)

print("Clasificador listo")

Modelo GPT-2 listo para inferencia
Cargando tokenizador lento de XLM-RoBERTa...
Cargando clasificador zero-shot (XLM-RoBERTa)...


Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Clasificador listo


In [25]:
print(is_protobuf_available())

True


In [27]:
categorias_candidatas = [
    "deportes",
    "política",
    "cultura",
    "economía",
    "tecnología",
    "salud",
    "entretenimiento",
    "internacional",
]

print(f"Clasificador listo - categorías: {categorias_candidatas}")



Clasificador listo - categorías: ['deportes', 'política', 'cultura', 'economía', 'tecnología', 'salud', 'entretenimiento', 'internacional']


In [29]:
# Funciones auxiliares de predicción
def _predecir_categoria_lote(textos: list[str], batch_size: int = 16) -> list[str]:
    """
    Predice la categoría periodística para una lista de textos
    usando zero-shot classification.
    """
    categorias_pred = []
    for i in tqdm(range(0, len(textos), batch_size), desc="Prediciendo categorías"):
        lote = textos[i:i + batch_size]
        resultados = clasificador_categoria(
            lote,
            candidate_labels=categorias_candidatas,
            multi_label=False,
            hypothesis_template="Este texto es sobre {}.",
        )

        if isinstance(resultados, dict):
            resultados = [resultados]

        for r in resultados:
            cat = r["labels"][0].upper().replace(" ", "_")
            categorias_pred.append(cat)

    return categorias_pred


def _predecir_sentimiento_lote(
    textos: list[str],
    categorias: list[str],
    batch_size: int = 32,
) -> tuple[list[int], list[float], list[float]]:
    """
    Predice el sentimiento para una lista de textos ya con su categoría.
    Devuelve: etiquetas_int, prob_positivo, prob_negativo.
    """
    model.eval()
    etiquetas, probs_pos, probs_neg = [], [], []

    for i in tqdm(range(0, len(textos), batch_size), desc="Prediciendo sentimiento"):
        lote_textos = textos[i:i + batch_size]
        lote_cats = categorias[i:i + batch_size]

        entradas = [
            f"[{cat.upper().replace(' ', '_')}] {txt}"
            for cat, txt in zip(lote_cats, lote_textos)
        ]

        inputs = tokenizer(
            entradas,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=max_len,
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits, dim=-1).cpu().numpy()

        for p in probs:
            clase = int(np.argmax(p))
            etiquetas.append(clase)
            probs_pos.append(round(float(p[0]) * 100, 2))
            probs_neg.append(round(float(p[1]) * 100, 2))

    return etiquetas, probs_pos, probs_neg

In [30]:
# Pipeline principal de producción
def pipeline_produccion(
    ruta_csv_entrada: str,
    ruta_csv_salida: str,
    columna_texto: str = "text",
    batch_size_zs: int = 16,
    batch_size_gpt: int = 32,
) -> pd.DataFrame:
    """
    Pipeline completo de inferencia en producción.

    Detecta automáticamente si el CSV ya tiene las columnas 'categoria'
    y/o 'negativo'. Si alguna falta o está vacía, la predice con el
    clasificador correspondiente.
    """
    print(f"Cargando: {ruta_csv_entrada}")
    df = pd.read_csv(ruta_csv_entrada)
    print(f"Registros: {len(df):,} | columnas: {list(df.columns)}")

    assert columna_texto in df.columns, (
        f"Error: la columna '{columna_texto}' no fue encontrada. "
        f"Columnas disponibles: {list(df.columns)}"
    )

    df[columna_texto] = df[columna_texto].astype(str).str.strip()
    textos = df[columna_texto].tolist()

    tiene_categoria = (
        "categoria" in df.columns
        and df["categoria"].notna().all()
        and (df["categoria"].astype(str).str.strip() != "").all()
    )
    tiene_negativo = (
        "negativo" in df.columns
        and df["negativo"].notna().all()
        and (df["negativo"].astype(str).str.strip() != "").all()
    )

    print(f"Columna 'categoria': {'presente' if tiene_categoria else 'ausente, se predecirá'}")
    print(f"Columna 'negativo': {'presente' if tiene_negativo else 'ausente, se predecirá'}")

    if not tiene_categoria:
        print("Etapa 1/2 - Predicción de categoría (zero-shot)...")
        categorias_pred = _predecir_categoria_lote(textos, batch_size=batch_size_zs)
        df["categoria"] = categorias_pred
        print("Distribución de categorías predichas:")
        print(df["categoria"].value_counts().to_string())
    else:
        df["categoria"] = (
            df["categoria"].astype(str).str.upper()
            .str.strip()
            .str.replace(r"\s+", "_", regex=True)
        )
        print("Usando categorías existentes (normalizadas)")

    if not tiene_negativo:
        print("Etapa 2/2 - Predicción de sentimiento (GPT-2)...")
        etiquetas, probs_pos, probs_neg = _predecir_sentimiento_lote(
            textos,
            df["categoria"].tolist(),
            batch_size=batch_size_gpt,
        )
        df["negativo"] = etiquetas
        df["sentimiento"] = df["negativo"].map({0: "Positivo", 1: "Negativo"})
        df["confianza_%"] = [
            probs_neg[i] if etiquetas[i] == 1 else probs_pos[i]
            for i in range(len(etiquetas))
        ]
        df["prob_positivo_%"] = probs_pos
        df["prob_negativo_%"] = probs_neg
    else:
        print("Columna 'negativo' ya presente, no se sobreescribe")

    df.to_csv(ruta_csv_salida, index=False, encoding="utf-8-sig")
    print(f"Resultado guardado en: {ruta_csv_salida}")
    print(f"Columnas finales: {list(df.columns)}")

    return df



In [32]:

sub_carpeta = 'datasets_clasificados'
nombre_archivo_clasificado = f'{nombre_archivo}_clasificado'

ruta_salida = os.path.join(sub_carpeta, f'{nombre_archivo_clasificado}.csv')
columna_de_texto = 'text'

df_resultado = pipeline_produccion(
    ruta_csv_entrada=ruta_entrada,
    ruta_csv_salida=ruta_salida,
    columna_texto=columna_de_texto,
    batch_size_zs=16,
    batch_size_gpt=32,
)

print("Vista previa")
display(
    df_resultado[["categoria", "negativo", "sentimiento", "confianza_%"]]
    .head(10)
)

print("Distribución final")
print(df_resultado["sentimiento"].value_counts().to_string())

print("Por categoría")
print(
    df_resultado.groupby("categoria")["negativo"]
    .agg(total="count", negativos="sum")
    .assign(pct_neg=lambda x: (x["negativos"] / x["total"] * 100).round(1))
    .sort_values("total", ascending=False)
    .to_string()
)

files.download(ruta_salida)
print(f"Descargando '{ruta_salida}'...")



Cargando: /home/papiit/DL_clasificador_post/datasets/sentiment_analysis_dataset_simulacion.csv
Registros: 2,584 | columnas: ['user', 'text', 'date', 'emotion', 'sentiment', 'Negative', 'Categoria']
Columna 'categoria': ausente, se predecirá
Columna 'negativo': ausente, se predecirá
Etapa 1/2 - Predicción de categoría (zero-shot)...


Prediciendo categorías:  65%|██████▌   | 106/162 [1:51:44<56:01, 60.02s/it]  

In [ ]:
# Evaluación global en Test
print("Evaluación en Test set")
preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = test_df[label_col].values

print("Métricas globales")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 (weighted): {f1_score(y_true, y_pred, average='weighted'):.4f}")

print("Reporte por clase")
print(classification_report(
    y_true,
    y_pred,
    target_names=['Positivo (0)', 'Negativo (1)'],
    digits=4,
))

cm = confusion_matrix(y_true, y_pred)
print("Matriz de confusión")
print(f"              Pred Pos  Pred Neg")
print(f"Real Pos :   {cm[0,0]:6d}    {cm[0,1]:6d}")
print(f"Real Neg :   {cm[1,0]:6d}    {cm[1,1]:6d}")



In [ ]:
# Evaluación desglosada por categoría periodística
test_df_eval = test_df.copy()
test_df_eval['pred'] = y_pred

print("=" * 60)
print("RENDIMIENTO POR CATEGORÍA PERIODÍSTICA")
print("=" * 60)

categorias = sorted(test_df_eval[category_col].unique())
resultados_cat = []

for cat in categorias:
    subset = test_df_eval[test_df_eval[category_col] == cat]
    if len(subset) < 5:
        continue

    yt = subset[label_col].values
    yp = subset['pred'].values

    acc = accuracy_score(yt, yp)
    f1 = f1_score(yt, yp, average='weighted', zero_division=0)
    n = len(subset)

    resultados_cat.append({
        'Categoría': cat,
        'N test': n,
        'Accuracy': round(acc, 4),
        'F1 (w)': round(f1, 4),
    })

if resultados_cat:
    df_resultados = pd.DataFrame(resultados_cat).sort_values('F1 (w)', ascending=False)
    print(df_resultados.to_string(index=False))

    peor = df_resultados.iloc[-1]
    print(f"Categoría con menor F1: {peor['Categoría']} ({peor['F1 (w)']})")
    print("Considera añadir más muestras de esa categoría al dataset.")
else:
    print("No hubo categorías con suficientes muestras para evaluar.")



In [ ]:
# Guardar modelo y tokenizador
save_path = './modelo_sentimientos_gpt2_es_v2'
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Modelo guardado en: {save_path}")

# Para descargar como ZIP:
# import shutil
# shutil.make_archive('modelo_sentimientos_gpt2_es_v2', 'zip', save_path)
# files.download('modelo_sentimientos_gpt2_es_v2.zip')

In [ ]:
# Función de inferencia lista para producción
def predecir_sentimiento(comentario: str, categoria: str) -> dict:
    """
    Predice el sentimiento de un comentario dado su categoría periodística.
    """
    model.eval()

    categoria_norm = categoria.upper().strip().replace(" ", "_")
    texto = f"[{categoria_norm}] {comentario}"

    inputs = tokenizer(
        texto,
        return_tensors='pt',
        truncation=True,
        padding='max_length',
        max_length=max_len,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    clase_pred = int(np.argmax(probs))

    etiquetas = {0: "Positivo", 1: "Negativo"}

    return {
        "texto": texto,
        "sentimiento": etiquetas[clase_pred],
        "confianza": f"{probs[clase_pred] * 100:.1f}%",
        "prob_positivo": f"{probs[0] * 100:.1f}%",
        "prob_negativo": f"{probs[1] * 100:.1f}%",
    }

# Prueba con ejemplos de distintas categorías
ejemplos = [
    ("El delantero anotó un gol magistral en el último minuto.", "deportes"),
    ("El árbitro tomó decisiones muy cuestionables durante todo el partido.", "deportes"),
    ("La nueva ley fue aprobada con amplio consenso legislativo.", "politica"),
    ("El presidente evadió todas las preguntas sobre el escándalo.", "politica"),
    ("El festival cultural superó todas las expectativas de asistencia.", "cultura"),
    ("La obra fue un total aburrimiento, no vale la pena ir.", "cultura"),
]

print("=" * 65)
print("PREDICCIONES DE EJEMPLO (con categoría)")
print("=" * 65)

for comentario, cat in ejemplos:
    r = predecir_sentimiento(comentario, cat)
    print(f"[{cat.upper()}]")
    print(comentario)
    print(f"-> {r['sentimiento']} (confianza: {r['confianza']})")